In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from typing import Tuple, Optional
import plotly.express as px
import plotly.graph_objects as go
from sklearn.mixture import GaussianMixture 

In [8]:
# Get data into dataframe
df = pd.read_csv('../../../WMA_fractions_v2.csv', skiprows=1)

# Preprocess data to only have temperature, salinity and dissolved oxygen
df_all = df.copy()
df_all = df_all[['Conservative_Temperature_[deg_C]', 'Absolute_Salinity_[PSU]', 'Depth_[m]', 'Latitude_[deg_N]', 'Longitude_[deg_E]']]
df_ST = df_all[['Conservative_Temperature_[deg_C]', 'Absolute_Salinity_[PSU]']]

In [9]:
# Define random seed for reproducibility
seed = 22

In [10]:
# Extract values into a tensor
X = torch.tensor(df_ST.values, dtype=torch.float32)
dataset = TensorDataset(X)
loader = DataLoader(dataset, batch_size=64, shuffle=True)

In [11]:
# Define Variational Autoencoder

class VAE(nn.Module):
    def __init__(self, input_dim=3, latent_dim=2):
        super().__init__()

        # Encoder
        self.fc1 = nn.Linear(input_dim, 16)
        self.fc_mu = nn.Linear(16, latent_dim)
        self.fc_logvar = nn.Linear(16, latent_dim)

        # Decoder
        self.fc2 = nn.Linear(latent_dim, 16)
        self.fc3 = nn.Linear(16, input_dim)

    def encode(self, x):
        h = F.relu(self.fc1(x))
        return self.fc_mu(h), self.fc_logvar(h)

    def reparameterize(self, mu, logvar):
        std = torch.exp(0.5 * logvar)
        eps = torch.randn_like(std)
        return mu + eps * std

    def decode(self, z):
        h = F.relu(self.fc2(z))
        return self.fc3(h)     # no sigmoid for real-valued data

    def forward(self, x):
        mu, logvar = self.encode(x)
        z = self.reparameterize(mu, logvar)
        return self.decode(z), mu, logvar

In [12]:
# Define loss function (mse and KL divergence)
def vae_loss(recon_x, x, mu, logvar):
    mse = F.mse_loss(recon_x, x, reduction='sum')
    kld = -0.5 * torch.sum(1 + logvar - mu.pow(2) - logvar.exp())
    return mse + kld

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"
model = VAE(input_dim=2, latent_dim=2).to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

epochs = 50

for epoch in range(epochs):
    model.train()
    total_loss = 0
    
    for batch in loader:
        x = batch[0].to(device)
        optimizer.zero_grad()

        recon, mu, logvar = model(x)
        loss = vae_loss(recon, x, mu, logvar)

        loss.backward()
        total_loss += loss.item()
        optimizer.step()

    print(f"Epoch {epoch+1} | Loss: {total_loss / len(dataset):.4f}")

Epoch 1 | Loss: 3.9778


In [ ]:
model.eval()
with torch.no_grad():
    mu, logvar = model.encode(X.to(device))
    z = mu.cpu()   # latent representation

In [ ]:
plt.scatter(z[:,0].cpu().tolist(), z[:,1].cpu().tolist(), marker='.')
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("Latent space")
plt.show()

In [ ]:
# Do BIC and AIC calculation for different number of clusters
bic_scores = []
n_components_range = range(1, 12)
for n_components in n_components_range:
    gmm = GaussianMixture(n_components=n_components, covariance_type='full', random_state=seed)
    gmm.fit(z.cpu().numpy())
    bic_scores.append(gmm.bic(z.cpu().numpy()))

In [ ]:
# BIC score plot
plt.plot(n_components_range, bic_scores, marker='o')
plt.xlabel("Number of components")  
plt.ylabel("BIC Score")
plt.title("BIC Scores for GMM with different number of components")
plt.show()

In [ ]:
with torch.no_grad():
    recon = model(X.to(device))[0].cpu().numpy()

In [ ]:
optimal_n_components = 4

In [ ]:
gmm = GaussianMixture(n_components=optimal_n_components, random_state=seed)
gmm.fit(z)
gmm_labels = gmm.predict(z)
df_all['gmm_label'] = gmm_labels.astype(int)

In [ ]:
plt.scatter(df_ST['Absolute_Salinity_[PSU]'], df_ST['Conservative_Temperature_[deg_C]'], c=df_all['gmm_label'], s=1, alpha=0.5)
plt.xlabel("Absolute Salinity [PSU]")
plt.ylabel("Temperature [C°]")
plt.title("GMM Clusters in T-S Space")
#plt.legend(np.unique(labels), loc='upper left', markerscale=5)
plt.legend(loc='upper right', labels=[f'Cluster {i}' for i in range(optimal_n_components)], markerscale=5)
plt.show()

In [ ]:
# Plot GMM on latent space with probability contours for each cluster
fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111)
scatter = ax.scatter(z[:,0].cpu().tolist(), z[:,1].cpu().tolist(), c=gmm_labels, cmap='viridis', marker='.')
plt.xlabel("z1")
plt.ylabel("z2")
plt.title("GMM Clustering in Latent Space")
plt.colorbar(scatter, label='GMM Cluster')
plt.show()

In [ ]:
# Temperature salinity diagram colored by GMM labels
fig5 = px.scatter(
    df_all,
    x='Absolute_Salinity_[PSU]',
    y='Conservative_Temperature_[deg_C]',
    color='gmm_label',
    title="Temperature-Salinity Diagram colored by GMM clusters"
)
fig5.update_layout(height=600, width=800)
fig5.show()

In [ ]:
# Compute data centroid (mean location)
center_lat = df_all['Latitude_[deg_N]'].mean()
center_lon = df_all['Longitude_[deg_E]'].mean()

print("Center lat:", center_lat, "Center lon:", center_lon)

# Use labels from kmeans clustering on data to plot geographically
fig3 = px.scatter_geo(
    df_all,
    lat='Latitude_[deg_N]',
    lon='Longitude_[deg_E]',
    color='gmm_label',
    title="Gaussian Mixture Model on VAE latent space projected on geographic map"
)

fig3.update_geos(
    projection_type="orthographic",
    projection_rotation=dict(lat=center_lat, lon=center_lon),
    showcoastlines=True,
    showcountries=True
)

fig3.update_layout(height=800, width=800)
fig3.show()

In [ ]:
# Plot in depth at different latitudes
fig4 = go.Figure()
for label in df_all['gmm_label'].unique():
    df_subset = df_all[df_all['gmm_label'] == label]
    fig4.add_trace(go.Scatter3d(
        x=df_subset['Longitude_[deg_E]'],
        y=df_subset['Latitude_[deg_N]'],
        z=df_subset['Depth_[m]'],
        mode='markers',
        marker=dict(size=2),
        name=f'Cluster {label}'
    ))
fig4.update_layout(
    scene=dict(
        xaxis_title='Longitude',
        yaxis_title='Latitude',
        zaxis_title='Depth (m)',
        zaxis=dict(autorange='reversed')  # Depth increases downwards
    ),
    title="3D Scatter plot of GMM clusters in geographic space"
)
fig4.show()